# Sanity check: training_table_avg10.csv

Run this **after** `build_training_table.ipynb`. It does two kinds of checks:

1. **Internal consistency** - no duplicate/null rows, `HOME_DIFF`/`HOME_WIN` agree, every
   `DIFF_*` column equals `HOME_* - AWAY_*`, values fall in plausible ranges, etc.
2. **Independent recomputation** - re-derives a sample of the rolling-window features
   directly from the raw per-game data (`data/raw/<season>/game_logs.csv` and
   `team_advanced_stats.csv`) using a different code path (explicit filter + `tail(N)` +
   `mean()`, instead of the `shift()` + `groupby().rolling()` used in
   `build_training_table.ipynb`) and checks they match what's in the CSV.

Every check either prints `OK: ...` or raises an `AssertionError` and stops the notebook -
if this notebook runs to the bottom without error, the training table passed every check
here.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import re

import numpy as np
import pandas as pd

from data_import import io_utils, settings

TABLE_PATH = settings.PROCESSED_DATA_DIR / "training_table_avg10.csv"

games = pd.read_csv(TABLE_PATH, dtype={"GAME_ID": str, "HOME_TEAM_ID": str, "AWAY_TEAM_ID": str})
games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"])

# Infer the trailing-window size from the column names instead of hardcoding it, so
# this notebook still works if the table is regenerated with a different N_GAMES.
avg_suffixes = {int(m.group(1)) for c in games.columns if (m := re.search(r"_AVG(\d+)$", c))}
assert len(avg_suffixes) == 1, f"expected one consistent _AVG<N> suffix, found {avg_suffixes}"
N_GAMES = avg_suffixes.pop()

print(f"loaded {len(games)} rows, {len(games.columns)} columns from {TABLE_PATH}")
print(f"detected trailing window: N_GAMES = {N_GAMES}")

loaded 4293 rows, 120 columns from /Users/jacobgipson/NBA Betting Pipeline/NBA_Betting_Pipeline/data/processed/training_table_avg10.csv
detected trailing window: N_GAMES = 10


## Structural checks

In [2]:
assert games["GAME_ID"].is_unique, "duplicate GAME_ID rows found"
print(f"OK: all {len(games)} GAME_ID values are unique")

null_counts = games.isna().sum()
bad_nulls = null_counts[null_counts > 0]
assert bad_nulls.empty, f"unexpected nulls:\n{bad_nulls}"
print("OK: no nulls in any column")

assert (games["HOME_TEAM_ID"] != games["AWAY_TEAM_ID"]).all(), "a game has the same team as home and away"
print("OK: HOME_TEAM_ID always differs from AWAY_TEAM_ID")

expected_seasons = set(settings.SEASONS)
found_seasons = set(games["SEASON"].unique())
assert found_seasons <= expected_seasons, f"unexpected season(s) in table: {found_seasons - expected_seasons}"
print(f"OK: seasons present {sorted(found_seasons)} are a subset of settings.SEASONS")

games.groupby("SEASON").size().rename("rows")

OK: all 4293 GAME_ID values are unique
OK: no nulls in any column
OK: HOME_TEAM_ID always differs from AWAY_TEAM_ID
OK: seasons present ['2022-23', '2023-24', '2024-25', '2025-26'] are a subset of settings.SEASONS


SEASON
2022-23    1072
2023-24    1074
2024-25    1075
2025-26    1072
Name: rows, dtype: int64

## Outcome consistency

In [3]:
assert (games["HOME_PTS"] != games["AWAY_PTS"]).all(), "found a tied game - NBA games can't end in a tie"
print("OK: no tied games (HOME_PTS != AWAY_PTS everywhere)")

expected_win = (games["HOME_PTS"] > games["AWAY_PTS"]).astype(int)
assert (games["HOME_WIN"] == expected_win).all(), "HOME_WIN disagrees with HOME_PTS > AWAY_PTS"
print("OK: HOME_WIN matches (HOME_PTS > AWAY_PTS)")

assert (games["HOME_DIFF"] == games["HOME_PTS"] - games["AWAY_PTS"]).all(), "HOME_DIFF != HOME_PTS - AWAY_PTS"
print("OK: HOME_DIFF == HOME_PTS - AWAY_PTS")

OK: no tied games (HOME_PTS != AWAY_PTS everywhere)
OK: HOME_WIN matches (HOME_PTS > AWAY_PTS)
OK: HOME_DIFF == HOME_PTS - AWAY_PTS


## HOME_DIFF vs HOME_WIN

`HOME_WIN` is just the sign of `HOME_DIFF` (1 when positive, 0 when negative), so the two
must agree in *every* row - that sign agreement (not a Pearson correlation of exactly
1.0) is what "perfectly correlated" means here. `HOME_DIFF` is a continuous margin and
`HOME_WIN` is binary, so even a perfectly consistent relationship won't show `corr() ==
1.0`; the point-biserial correlation is still reported below for reference and should be
strongly positive.

In [4]:
sign_matches_win = (games["HOME_DIFF"] > 0) == (games["HOME_WIN"] == 1)
assert sign_matches_win.all(), f"HOME_DIFF sign disagrees with HOME_WIN in {(~sign_matches_win).sum()} row(s)"
print("OK: sign(HOME_DIFF) agrees with HOME_WIN in every row (perfectly correlated)")

point_biserial_r = games["HOME_DIFF"].corr(games["HOME_WIN"])
print(f"point-biserial correlation(HOME_DIFF, HOME_WIN) = {point_biserial_r:.4f}")
assert point_biserial_r > 0.5, "correlation between HOME_DIFF and HOME_WIN is surprisingly weak"


OK: sign(HOME_DIFF) agrees with HOME_WIN in every row (perfectly correlated)
point-biserial correlation(HOME_DIFF, HOME_WIN) = 0.7979


## Rest-day checks

In [5]:
assert (games[["HOME_REST", "AWAY_REST"]] >= 0).all().all(), "found a negative rest value"
print("OK: HOME_REST and AWAY_REST are never negative")

assert (games["REST_DIFF"] == games["HOME_REST"] - games["AWAY_REST"]).all(), "REST_DIFF != HOME_REST - AWAY_REST"
print("OK: REST_DIFF == HOME_REST - AWAY_REST")

OK: HOME_REST and AWAY_REST are never negative
OK: REST_DIFF == HOME_REST - AWAY_REST


## Every `DIFF_*` column reconstructs `HOME_* - AWAY_*`

In [6]:
diff_cols = [c for c in games.columns if c.startswith("DIFF_")]
checked = 0
for diff_col in diff_cols:
    stat = diff_col[len("DIFF_"):]
    home_col, away_col = f"HOME_{stat}", f"AWAY_{stat}"
    if home_col not in games.columns or away_col not in games.columns:
        continue
    matches = np.isclose(games[diff_col], games[home_col] - games[away_col])
    assert matches.all(), f"{diff_col} != {home_col} - {away_col} in {(~matches).sum()} row(s)"
    checked += 1

assert checked > 0, "no DIFF_* columns were actually checked - column-name pattern may have changed"
print(f"OK: all {checked} DIFF_* columns equal their HOME_*/AWAY_* difference")

OK: all 35 DIFF_* columns equal their HOME_*/AWAY_* difference


## Value-range sanity checks

Loose plausibility bounds on rolling averages - not tight enough to catch subtle bugs,
but would catch a units error, a badly broken merge, or a rate stat computed on the
wrong denominator.

In [7]:
range_checks = {
    f"HOME_PTS_AVG{N_GAMES}": (70, 150),
    f"AWAY_PTS_AVG{N_GAMES}": (70, 150),
    f"HOME_FG_PCT_AVG{N_GAMES}": (0.30, 0.60),
    f"AWAY_FG_PCT_AVG{N_GAMES}": (0.30, 0.60),
    f"HOME_WIN_AVG{N_GAMES}": (0, 1),
    f"AWAY_WIN_AVG{N_GAMES}": (0, 1),
    f"HOME_TS_PCT_AVG{N_GAMES}": (0.40, 0.70),
    f"AWAY_TS_PCT_AVG{N_GAMES}": (0.40, 0.70),
    f"HOME_OFF_RATING_AVG{N_GAMES}": (90, 140),
    f"AWAY_OFF_RATING_AVG{N_GAMES}": (90, 140),
    f"HOME_DEF_RATING_AVG{N_GAMES}": (90, 140),
    f"AWAY_DEF_RATING_AVG{N_GAMES}": (90, 140),
    "HOME_REST": (0, 30),
    "AWAY_REST": (0, 30),
    "HOME_SPREAD": (-40, 40),
}

for col, (lo, hi) in range_checks.items():
    assert col in games.columns, f"expected column {col} not found"
    out_of_range = games[(games[col] < lo) | (games[col] > hi)]
    assert out_of_range.empty, f"{col} has {len(out_of_range)} value(s) outside [{lo}, {hi}]:\n{out_of_range[['GAME_ID', col]]}"

print(f"OK: all {len(range_checks)} spot-checked columns fall within plausible ranges")

OK: all 15 spot-checked columns fall within plausible ranges


## Spread sanity checks

`HOME_SPREAD` is the closing line from the home team's perspective (negative = home
favored, positive = home underdog; see `docs/training_table_columns.md`). Three checks
that would catch a sign flip, a bad merge, or garbage odds data:

1. The spread favorite (regardless of which side it's on) should win the large majority
   of games - we use **55%** as the low bar.
2. Win rate should decrease monotonically as `HOME_SPREAD` moves from "home heavily
   favored" to "home heavily the underdog".
3. `HOME_SPREAD` and `HOME_DIFF` (actual margin) should be strongly negatively
   correlated, since a more negative spread means a bigger expected home margin.

In [8]:
favored_home = games["HOME_SPREAD"] < 0
favored_away = games["HOME_SPREAD"] > 0
pickem = games["HOME_SPREAD"] == 0

favorite_won = pd.concat([
    games.loc[favored_home, "HOME_WIN"] == 1,
    games.loc[favored_away, "HOME_WIN"] == 0,
])
favorite_win_rate = favorite_won.mean()
print(
    f"favorite win rate: {favorite_win_rate:.1%} over {len(favorite_won)} games "
    f"({pickem.sum()} pick'em game(s) with HOME_SPREAD == 0 excluded)"
)
assert favorite_win_rate > 0.55, f"favorite win rate {favorite_win_rate:.1%} is suspiciously low (expected > 55%)"
print("OK: the spread favorite wins more than 55% of the time")

favorite win rate: 69.1% over 4282 games (11 pick'em game(s) with HOME_SPREAD == 0 excluded)
OK: the spread favorite wins more than 55% of the time


In [9]:
# Home win rate should climb monotonically as HOME_SPREAD improves for the home team
# (goes from underdog to bigger and bigger favorite).
spread_bins = pd.cut(
    games["HOME_SPREAD"],
    bins=[-100, -10, -5, 0, 5, 10, 100],
    labels=["home -10 or more", "home -10 to -5", "home -5 to 0", "home 0 to 5", "home 5 to 10", "home 10+"],
)
win_rate_by_bin = games.groupby(spread_bins, observed=True)["HOME_WIN"].mean()
print(win_rate_by_bin)

assert win_rate_by_bin.is_monotonic_decreasing, (
    "home win rate should decrease monotonically as HOME_SPREAD increases (home team going "
    "from bigger favorite to bigger underdog)"
)
print("OK: home win rate decreases monotonically as HOME_SPREAD gets worse for the home team")

HOME_SPREAD
home -10 or more    0.839470
home -10 to -5      0.721457
home -5 to 0        0.567245
home 0 to 5         0.421986
home 5 to 10        0.269091
home 10+            0.132143
Name: HOME_WIN, dtype: float64
OK: home win rate decreases monotonically as HOME_SPREAD gets worse for the home team


In [10]:
spread_margin_corr = games["HOME_SPREAD"].corr(games["HOME_DIFF"])
print(f"correlation(HOME_SPREAD, HOME_DIFF) = {spread_margin_corr:.4f}")
assert spread_margin_corr < -0.4, (
    f"HOME_SPREAD should be strongly negatively correlated with HOME_DIFF, got {spread_margin_corr:.4f}"
)
print("OK: HOME_SPREAD is strongly negatively correlated with HOME_DIFF (more negative spread -> bigger home margin)")

correlation(HOME_SPREAD, HOME_DIFF) = -0.5046
OK: HOME_SPREAD is strongly negatively correlated with HOME_DIFF (more negative spread -> bigger home margin)


## Independent recomputation of rolling-window stats

Rebuilds a team-game table straight from the raw CSVs (mirroring only the unavoidable
prep steps from `build_training_table.ipynb` - the opponent-points self-join and the
neutral-site-game drop - since those are data-cleaning steps, not the rolling-window
logic being tested) and, for a random sample of games, recomputes each team's trailing
`N_GAMES`-game average with `sort_values("GAME_DATE").tail(N_GAMES)` instead of
`shift()`/`rolling()`. Covers a mix of plain count averages, a shooting-percentage rate
stat, a win-rate stat, and the two possession-based ratings - 10 stats in total.

In [11]:
raw_logs = pd.concat(
    [io_utils.load_existing(season, "game_logs").assign(SEASON=season) for season in settings.SEASONS],
    ignore_index=True,
)
raw_logs["GAME_DATE"] = pd.to_datetime(raw_logs["GAME_DATE"])
raw_logs["WIN"] = (raw_logs["WL"] == "W").astype(int)
raw_logs["IS_HOME"] = raw_logs["IS_HOME"].astype(bool)

# Same neutral-site-game drop as build_training_table.ipynb, so a team's trailing
# window lines up game-for-game with the training table.
home_counts = raw_logs.groupby(["SEASON", "GAME_ID"])["IS_HOME"].transform("sum")
raw_logs = raw_logs[home_counts > 0]

opp_pts = raw_logs[["GAME_ID", "TEAM_ID", "PTS"]].rename(columns={"TEAM_ID": "OPP_TEAM_ID", "PTS": "OPP_PTS"})
raw_logs = raw_logs.merge(opp_pts, on="GAME_ID")
raw_logs = raw_logs[raw_logs["TEAM_ID"] != raw_logs["OPP_TEAM_ID"]].drop(columns=["OPP_TEAM_ID"])

raw_adv = pd.concat(
    [io_utils.load_existing(season, "team_advanced_stats").assign(SEASON=season) for season in settings.SEASONS],
    ignore_index=True,
)

raw = raw_logs.merge(
    raw_adv[["SEASON", "GAME_ID", "TEAM_ID", "POSS"]],
    on=["SEASON", "GAME_ID", "TEAM_ID"],
    how="inner",
)
print(f"raw per-team-game rows available for cross-checking: {len(raw)}")

raw per-team-game rows available for cross-checking: 9836


In [12]:
def trailing_window(team_id, season, game_date, n=N_GAMES):
    prior = raw[
        (raw["SEASON"] == season) & (raw["TEAM_ID"] == team_id) & (raw["GAME_DATE"] < game_date)
    ]
    return prior.sort_values("GAME_DATE").tail(n)


STAT_TO_COLUMN = {
    "PTS_AVG": "PTS",
    "REB_AVG": "REB",
    "AST_AVG": "AST",
    "TOV_AVG": "TOV",
    "STL_AVG": "STL",
    "FGM_AVG": "FGM",
    "WIN_AVG": "WIN",
    "FG_PCT_AVG": "FG_PCT",
    "OFF_RATING_AVG": "OFF_RATING",
    "DEF_RATING_AVG": "DEF_RATING",
}


def recompute_stats(prior):
    return {
        "PTS_AVG": prior["PTS"].mean(),
        "REB_AVG": prior["REB"].mean(),
        "AST_AVG": prior["AST"].mean(),
        "TOV_AVG": prior["TOV"].mean(),
        "STL_AVG": prior["STL"].mean(),
        "FGM_AVG": prior["FGM"].mean(),
        "WIN_AVG": prior["WIN"].mean(),
        "FG_PCT_AVG": prior["FGM"].sum() / prior["FGA"].sum(),
        "OFF_RATING_AVG": 100 * prior["PTS"].sum() / prior["POSS"].sum(),
        "DEF_RATING_AVG": 100 * prior["OPP_PTS"].sum() / prior["POSS"].sum(),
    }

In [13]:
rng = np.random.default_rng(0)
sample_idx = rng.choice(games.index, size=min(40, len(games)), replace=False)

mismatches = []
checked_pairs = 0

for idx in sample_idx:
    row = games.loc[idx]
    for side in ["HOME", "AWAY"]:
        team_id = row[f"{side}_TEAM_ID"]
        prior = trailing_window(team_id, row["SEASON"], row["GAME_DATE"])
        assert len(prior) == N_GAMES, (
            f"expected {N_GAMES} prior games for team {team_id} before {row['GAME_DATE'].date()}, "
            f"found {len(prior)} - this training table row should have been dropped"
        )
        recomputed = recompute_stats(prior)
        for stat_name, value in recomputed.items():
            col = f"{side}_{STAT_TO_COLUMN[stat_name]}_AVG{N_GAMES}"
            table_value = row[col]
            checked_pairs += 1
            if not np.isclose(value, table_value, rtol=1e-6, atol=1e-6):
                mismatches.append((row["GAME_ID"], col, value, table_value))

assert not mismatches, f"{len(mismatches)} mismatch(es) vs. the training table, e.g. {mismatches[:5]}"
print(
    f"OK: {checked_pairs} independently recomputed (stat, team) values across "
    f"{len(sample_idx)} sampled games all match the training table"
)
print(f"stats cross-checked: {sorted(STAT_TO_COLUMN.values())}")

OK: 800 independently recomputed (stat, team) values across 40 sampled games all match the training table
stats cross-checked: ['AST', 'DEF_RATING', 'FGM', 'FG_PCT', 'OFF_RATING', 'PTS', 'REB', 'STL', 'TOV', 'WIN']


## Independent recomputation of rest days

In [14]:
def recompute_rest(team_id, season, game_date):
    prior_dates = raw.loc[
        (raw["SEASON"] == season) & (raw["TEAM_ID"] == team_id) & (raw["GAME_DATE"] < game_date),
        "GAME_DATE",
    ]
    assert not prior_dates.empty, f"no prior game found for team {team_id} before {game_date.date()}"
    return (game_date - prior_dates.max()).days - 1


rest_mismatches = []
for idx in sample_idx:
    row = games.loc[idx]
    for side in ["HOME", "AWAY"]:
        team_id = row[f"{side}_TEAM_ID"]
        recomputed_rest = recompute_rest(team_id, row["SEASON"], row["GAME_DATE"])
        table_rest = row[f"{side}_REST"]
        if recomputed_rest != table_rest:
            rest_mismatches.append((row["GAME_ID"], side, recomputed_rest, table_rest))

assert not rest_mismatches, f"{len(rest_mismatches)} rest-day mismatch(es), e.g. {rest_mismatches[:5]}"
print(f"OK: recomputed rest days match HOME_REST/AWAY_REST for all {len(sample_idx)} sampled games (both sides)")

OK: recomputed rest days match HOME_REST/AWAY_REST for all 40 sampled games (both sides)


## Summary

In [15]:
print("All sanity checks passed.")
print(f"{TABLE_PATH} is internally consistent and matches an independent recomputation from raw data.")

All sanity checks passed.
/Users/jacobgipson/NBA Betting Pipeline/NBA_Betting_Pipeline/data/processed/training_table_avg10.csv is internally consistent and matches an independent recomputation from raw data.
